In [1]:
import os
from typing import Annotated, Sequence, TypedDict
from dotenv import load_dotenv  
from langchain_core.messages import BaseMessage # The foundational class for all message types in LangGraph
from langchain_core.messages import SystemMessage # Message for providing instructions to the LLM
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.tools import tool
from langgraph.graph.message import add_messages
from langgraph.graph import StateGraph, END
from langgraph.prebuilt import ToolNode


load_dotenv()

google_api_key = os.environ["GOOGLE_API_KEY"]

free_model_names = [
    # Current stable Gemini models
    "gemini-3.8-flash",
    "gemini-3.7-flash",
    "gemini-3.6-flash",
    "gemini-3.5-flash",
    "gemini-3.5-flash-lite",
    "gemini-3.1-flash-lite",

    # Preview models
    "gemini-3-flash-preview",

    # Legacy models - restricted/deprecated
    "gemini-2.5-flash",
    "gemini-2.5-flash-lite",
    "gemini-2.5-pro",
]

models = [
    ChatGoogleGenerativeAI(
        model=model_name,
        google_api_key=google_api_key,
        max_retries=0,
        timeout=30,
    )
    for model_name in free_model_names
]

In [4]:
llm = models[0].with_fallbacks(models[1:])
response = llm.invoke("why do parrots talk?")
print(response.response_metadata)
print(response.content)

{'finish_reason': 'STOP', 'model_name': 'gemini-3.6-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}
[{'type': 'text', 'text': 'Parrots don’t "talk" in the way humans do—they don\'t naturally construct sentences to express original thoughts. Instead, they are **master mimics**. \n\nThere are three main reasons why parrots mimic human speech: **social bonding, physical ability, and high intelligence.**\n\n---\n\n### 1. Social Bonding (The "Flock" Mentality)\nIn the wild, parrots are extremely social flock animals. Communication is essential for their survival. \n* **Flock Dialects:** Wild parrots learn specific vocalizations to communicate with their group. Different flocks actually have different "dialects."\n* **Humans as the Flock:** When a parrot lives in a human home, it views the humans (and sometimes other pets) as its flock. Because parrots naturally want to fit in and communicate with their group, they copy the noises they hear most often—human words, laughter, d

# streaming is the process of printing the data while it is being generated

In [5]:
for chunk in llm.stream("Why do parrots have colorful feathers?"):
    print(chunk.text, end="|", flush=True)

print(response.response_metadata.get("model_name"))

Parrots have colorful feathers| due to a combination of **evolutionary survival strategies, social communication, and unique biology**. While bright red, yellow, blue|, and green feathers might make a parrot stand out to human eyes, these colors serve vital purposes in the wild.

Here| are the main reasons why parrots are so colorful:

---

### 1. Camouflage in the Wild
It may| seem counterintuitive, but bright colors actually help parrots hide. 
* **The Rainforest Canopy:** Most parrots live in dense|, tropical rainforests filled with bright sunlight, deep shadows, colorful flowers, fruits, and lush green leaves. 
*| **Disruptive Coloration:** A bright green, red, or yellow parrot sitting in a tree breaks up the bird's silhouette. To| predators (like hawks or snakes) looking up into a canopy broken by dappled sunlight and colorful flora, the parrot blends right| into the background.

---

### 2. Mating and Sexual Selection
Feather color plays a massive role in choosing a| mate:
* **He

# batch is the process of passing multiple in one promt

In [6]:
responses = llm.batch([
    "Why do parrots have colorful feathers?",
    "How do airplanes fly?",
    "What is quantum computing?"
])
for response in responses:
    print(response)

content=[{'type': 'text', 'text': "Parrots have colorful feathers due to a combination of **evolutionary survival strategies, social communication, and unique biology**. While bright colors might seem like they would make a bird easy for predators to spot, in the wild, these colors serve several vital purposes.\n\nHere are the main reasons why parrots are so colorful:\n\n---\n\n### 1. Camouflage in Tropical Environments\nIt may seem counterintuitive, but bright colors actually help parrots blend in. \n* **Leafy Canopies:** Most parrots live in dense, tropical rainforests. Bright green feathers allow them to blend seamlessly into the lush foliage.\n* **Sunlight and Blossoms:** Bright reds, yellows, and blues mimic the dappled sunlight, bright flowers, and colorful fruits found high up in the tree canopy. \n* **Disruptive Coloration:** Bold, contrasting patches of color break up the outline of the parrot’s body, making it hard for predators (like hawks or snakes) to recognize the bird as